# Bridge Diagnostic
Step-by-step inspection: T_eng predictor → bridge ODE → Whisper decode.

In [1]:
import os, sys, json, torch
import numpy as np
from pathlib import Path

os.environ.setdefault('TRAIN_DATA_DIR', '/vol/gpudata/tsv22-train/data/processed/train')
os.environ.setdefault('DEV_DATA_DIR',   '/vol/gpudata/tsv22-dev_test/data/processed/dev')
os.environ.setdefault('TEST_DATA_DIR',  '/vol/gpudata/tsv22-dev_test/data/processed/test')

ROOT = '/vol/gpudata/tsv22-fyp/accent-robust-asr'
sys.path.insert(0, ROOT)
os.chdir(ROOT)

BRIDGE_CKPT    = 'models/bridge_dtw_eps_0.5/checkpoint_best.pt'
PREDICTOR_CKPT = 'models/tnat_predictor/model_best.pt'

PAIR = {
    "l2_speaker":            "THV",
    "l2_utterance_id":       "arctic_a0313",
    "l2_encoder_state_path": "THV/THV_arctic_a0313.pt",
    "nat_speaker":           "CLB",
    "nat_encoder_state_path":"CLB/CLB_arctic_a0313.pt",
    "l2_speech_end_frame":   193,
    "nat_speech_end_frame":  162,
    "text":                  "Broken Tooth yelled with fright and pain",
    "l1":                    "Vietnamese",
}

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_STEPS = 20  # ODE steps — change here to update throughout notebook

# Read training config — drives all alignment-specific behaviour downstream
_cfg             = json.loads((Path(BRIDGE_CKPT).parent / 'config.json').read_text())
ALIGNMENT        = _cfg['alignment']         # 'dtw_fixed' | 'dtw' | ...
SIGMA_MAX        = float(_cfg['sigma_max'])
PARAMETERIZATION = _cfg['parameterization']

print(f'device: {device}')
print(f'Utterance: "{PAIR["text"]}"  |  l2={PAIR["l2_speaker"]}  nat={PAIR["nat_speaker"]}')
print(f'alignment={ALIGNMENT}  sigma_max={SIGMA_MAX}  parameterization={PARAMETERIZATION}')

device: cuda
Utterance: "Broken Tooth yelled with fright and pain"  |  l2=THV  nat=CLB
alignment=dtw  sigma_max=0.5  parameterization=eps


In [2]:
import numpy as np

# ── Alignment registry ────────────────────────────────────────────────────────
# Each mode exposes:
#   uses_predictor  — whether a T_nat predictor is needed
#   n_frames(t, T_l2, T_nat) → int — active speech frames at bridge timestep t
#   make_oracle(path, z_acc, z_nat, T_l2, T_nat)
#       → (oracle_fn, z_nat_warped | None)
#       oracle_fn(t) → (z_interp [1500, D], N_t)  on-manifold interpolation
#       path may be None for non-DTW modes
# ─────────────────────────────────────────────────────────────────────────────

def _n_frames_fixed(t, T_l2, T_nat):
    return T_l2

def _n_frames_morph(t, T_l2, T_nat):
    return max(1, round(float(t) * T_l2 + (1 - float(t)) * T_nat))


# ── Position oracles (no DTW path needed) ────────────────────────────────────

def _make_oracle_position_fixed(path, z_acc, z_nat, T_l2, T_nat):
    """Fixed-timeline oracle: straight-line blend on L2 grid, no morphing."""
    z_nat_trunc = z_nat[:T_l2].clone()  # position-aligned native target

    def oracle(t):
        z = z_acc.clone()
        z[:T_l2] = float(t) * z_acc[:T_l2] + (1 - float(t)) * z_nat_trunc
        return z, T_l2

    return oracle, z_nat_trunc  # analogous to z_nat_warped


def _make_oracle_position(path, z_acc, z_nat, T_l2, T_nat):
    """Position oracle: straight-line blend with N(t) morphing."""
    def oracle(t):
        if t == 1.0:
            return z_acc.clone(), T_l2
        if t == 0.0:
            z = z_acc.clone()
            z[:T_nat] = z_nat[:T_nat]
            return z, T_nat
        N_t    = max(1, round(float(t) * T_l2 + (1 - float(t)) * T_nat))
        speech = float(t) * z_acc[:N_t] + (1 - float(t)) * z_nat[:N_t]
        z      = z_acc.clone()
        z[:N_t] = speech
        return z, N_t

    return oracle, None


# ── DTW oracles ───────────────────────────────────────────────────────────────

def _make_oracle_dtw_fixed(path, z_acc, z_nat, T_l2, T_nat):
    """Fixed-timeline oracle: each L2 frame k maps to its DTW-matched native frame."""
    P      = len(path)
    j_norm = path[:, 1].astype(np.float32) / max(T_l2 - 1, 1)
    k_grid = np.linspace(0.0, 1.0, T_l2, dtype=np.float32)
    idx_r  = np.clip(np.searchsorted(j_norm, k_grid), 0, P - 1)
    idx_l  = np.clip(idx_r - 1, 0, P - 1)
    k_idx  = np.where(
        np.abs(j_norm[idx_l] - k_grid) <= np.abs(j_norm[idx_r] - k_grid),
        idx_l, idx_r,
    )
    z_nat_warped = z_nat[path[k_idx, 0].astype(int)].clone()  # [T_l2, D]

    def oracle(t):
        z = z_acc.clone()
        z[:T_l2] = float(t) * z_acc[:T_l2] + (1 - float(t)) * z_nat_warped
        return z, T_l2

    return oracle, z_nat_warped


def _make_oracle_dtw(path, z_acc, z_nat, T_l2, T_nat):
    """Alpha-timeline oracle with N(t) morphing."""
    def oracle(t):
        if t == 1.0:
            return z_acc.clone(), T_l2
        if t == 0.0:
            z = z_acc.clone()
            z[:T_nat] = z_nat[:T_nat]
            return z, T_nat
        N_t    = max(1, round(float(t) * T_l2 + (1 - float(t)) * T_nat))
        i_norm = path[:, 0].astype(np.float32) / max(T_nat - 1, 1)
        j_norm = path[:, 1].astype(np.float32) / max(T_l2  - 1, 1)
        t_k    = float(t) * j_norm + (1 - float(t)) * i_norm
        out_t  = np.linspace(0.0, 1.0, N_t, dtype=np.float32)
        idx_r  = np.clip(np.searchsorted(t_k, out_t), 0, len(t_k) - 1)
        idx_l  = np.clip(idx_r - 1, 0, len(t_k) - 1)
        k_idx  = np.where(
            np.abs(t_k[idx_l] - out_t) <= np.abs(t_k[idx_r] - out_t),
            idx_l, idx_r,
        )
        speech = float(t) * z_acc[path[k_idx, 1].astype(int)] + \
                 (1 - float(t)) * z_nat[path[k_idx, 0].astype(int)]
        z = z_acc.clone()
        z[:N_t] = speech
        return z, N_t

    return oracle, None


ALIGNMENT_MODES = {
    "position": {
        "uses_predictor": True,
        "n_frames":       _n_frames_morph,
        "make_oracle":    _make_oracle_position,
    },
    "position_fixed": {
        "uses_predictor": False,
        "n_frames":       _n_frames_fixed,
        "make_oracle":    _make_oracle_position_fixed,
    },
    "dtw": {
        "uses_predictor": True,
        "n_frames":       _n_frames_morph,
        "make_oracle":    _make_oracle_dtw,
    },
    "dtw_fixed": {
        "uses_predictor": False,
        "n_frames":       _n_frames_fixed,
        "make_oracle":    _make_oracle_dtw_fixed,
    },
}


def recover_x0(z_t_crop, model_out, t_cur, sigma_max, parameterization):
    """Recover z_nat_hat from raw model output (eps or x0 only; cfm uses Euler)."""
    if parameterization == "x0":
        return model_out
    if parameterization == "eps":
        return z_t_crop - sigma_max * float(t_cur) ** 0.5 * model_out
    raise ValueError(f"recover_x0 not defined for {parameterization!r}")


MODE = ALIGNMENT_MODES[ALIGNMENT]
print(f'Active mode : {ALIGNMENT}')
print(f'uses_predictor={MODE["uses_predictor"]}  n_frames={"fixed" if not MODE["uses_predictor"] else "morphing"}')

Active mode : dtw
uses_predictor=True  n_frames=morphing


## 1. Load encoder state

In [3]:
from pathlib import Path

dev_dir   = Path(os.environ['DEV_DATA_DIR'])
train_dir = Path(os.environ['TRAIN_DATA_DIR'])

def load_state(rel_path):
    for d in [dev_dir, train_dir]:
        p = d / rel_path
        if p.exists():
            s = torch.load(p, map_location='cpu', weights_only=False)
            return s['hidden_states'].float(), p
    return None, None

z_acc, p_acc = load_state(PAIR['l2_encoder_state_path'])
z_nat, p_nat = load_state(PAIR['nat_encoder_state_path'])

assert z_acc is not None, f"z_acc not found: {PAIR['l2_encoder_state_path']}"
assert z_nat is not None, f"z_nat not found: {PAIR['nat_encoder_state_path']}"

T_l2     = PAIR['l2_speech_end_frame']
T_nat_gt = PAIR['nat_speech_end_frame']  # ground-truth T_nat from annotation

print(f'z_acc: {p_acc}')
print(f'z_nat: {p_nat}')
print(f'T_l2={T_l2}  T_nat_gt={T_nat_gt}')
print(f'z_acc speech scale: {z_acc[:T_l2].norm(dim=-1).mean():.3f}')
print(f'z_nat speech scale: {z_nat[:T_nat_gt].norm(dim=-1).mean():.3f}')
cos_baseline = torch.nn.functional.cosine_similarity(
    z_acc[:min(T_l2, T_nat_gt)], z_nat[:min(T_l2, T_nat_gt)], dim=-1).mean()
print(f'Baseline cos_sim(z_acc, z_nat): {cos_baseline:.4f}')

z_acc: /vol/gpudata/tsv22-train/data/processed/train/THV/THV_arctic_a0313.pt
z_nat: /vol/gpudata/tsv22-train/data/processed/train/CLB/CLB_arctic_a0313.pt
T_l2=193  T_nat_gt=162
z_acc speech scale: 41.278
z_nat speech scale: 41.411
Baseline cos_sim(z_acc, z_nat): 0.4803


## 2. T_nat — predictor (dtw) or fixed to T_l2 (dtw_fixed)

In [4]:
if MODE['uses_predictor']:
    from src.experiments.exp2_latent_diffusion_bridge.eval import load_tnat_predictor
    from src.experiments.exp2_latent_diffusion_bridge.train_tnat_predictor import T_NORM
    predictor  = load_tnat_predictor(PREDICTOR_CKPT, device)
    z_acc_dev  = z_acc.to(device)
    with torch.no_grad():
        pool       = z_acc_dev[:T_l2].mean(dim=0, keepdim=True)
        t_l2_n     = torch.tensor([T_l2 / T_NORM], device=device)
        T_nat_raw  = predictor(pool, t_l2_n).item() * T_NORM
    T_nat = max(1, min(int(round(T_nat_raw)), 1500))
    print(f'T_l2={T_l2}  T_nat_hat={T_nat}  (raw={T_nat_raw:.1f})  T_nat_gt={T_nat_gt}')
    print(f'inf_len={max(T_l2, T_nat) + 1}')
else:
    T_nat = T_l2  # dtw_fixed: bridge is trained on L2 timeline, T_nat = T_l2
    print(f'{ALIGNMENT!r}: T_nat fixed to T_l2={T_l2}  (no predictor needed)')

T_l2=193  T_nat_hat=173  (raw=172.9)  T_nat_gt=162
inf_len=194


## 3. Bridge ODE — inspect z_nat_hat

In [5]:
from src.experiments.exp2_latent_diffusion_bridge.eval import load_bridge_model
from src.experiments.exp2_latent_diffusion_bridge.diffusion import bridge_inference

bridge     = load_bridge_model(BRIDGE_CKPT, device)
z_acc_bf16 = z_acc.to(device=device, dtype=torch.bfloat16).unsqueeze(0)  # [1, 1500, 768]

print(f'alignment={ALIGNMENT}  parameterization={PARAMETERIZATION}  sigma_max={SIGMA_MAX}')

with torch.no_grad():
    z_nat_hat = bridge_inference(
        bridge, z_acc_bf16, T_l2=T_l2, T_nat=T_nat,
        n_steps=N_STEPS, sigma_max=SIGMA_MAX, parameterization=PARAMETERIZATION,
    )

z_nat_hat_cpu = z_nat_hat.squeeze(0).float().cpu()

print(f'\nz_acc  (0:{T_l2}):  scale={z_acc[:T_l2].norm(dim=-1).mean():.3f}')
print(f'z_hat  (0:{T_nat}): scale={z_nat_hat_cpu[:T_nat].norm(dim=-1).mean():.3f}')
print(f'z_hat tail ({T_nat}:): scale={z_nat_hat_cpu[T_nat:T_nat+20].norm(dim=-1).mean():.3f}')
print(f'NaN — z_acc={z_acc.isnan().any()}  z_hat={z_nat_hat_cpu.isnan().any()}')
n = min(T_l2, T_nat)
print(f'\ncos(z_hat, z_acc) over {n} speech frames: '
      f'{torch.nn.functional.cosine_similarity(z_acc[:n], z_nat_hat_cpu[:n], dim=-1).mean():.4f}')

[Eval] Baked EMA-averaged weights into model from models/bridge_dtw_eps_0.5/checkpoint_best.pt
alignment=dtw  parameterization=eps  sigma_max=0.5

z_acc  (0:193):  scale=41.278
z_hat  (0:173): scale=41.733
z_hat tail (173:): scale=37.851
NaN — z_acc=False  z_hat=False

cos(z_hat, z_acc) over 173 speech frames: 0.9058


## 4. Per-step ODE trace — watch how z_t evolves

In [6]:
B, L, D    = z_acc_bf16.shape
dtype      = z_acc_bf16.dtype
sil        = z_acc_bf16[:, T_l2:T_l2 + 1, :].clone()
inf_len    = max(T_l2, T_nat) + 1
z_acc_crop = z_acc_bf16[:, :inf_len, :]
t_schedule = torch.linspace(1.0, 0.0, N_STEPS + 1, device=device, dtype=dtype)
is_cfm     = PARAMETERIZATION == 'cfm'

def apply_mask(z, t):
    N_t = MODE['n_frames'](t, T_l2, T_nat)
    z[:, N_t:, :] = sil.expand(B, L - N_t, D)
    return z, N_t

z_t = z_acc_bf16.clone()
z_t, _ = apply_mask(z_t, 1.0)

bridge.eval()
print(f'{"step":>4}  {"t":>5}  {"N(t)":>5}  {"z_t_scale":>10}  {"z_hat_scale":>11}  {"vel_norm":>10}  {"vel_step":>10}')
with torch.no_grad():
    for i in range(N_STEPS):
        t_cur  = t_schedule[i]
        t_next = t_schedule[i + 1]
        dt     = (t_cur - t_next).item()
        _, N_t = apply_mask(z_t.clone(), t_cur.item())

        t_batch  = torch.full((B,), t_cur, device=device, dtype=dtype)
        out_crop = bridge(z_t[:, :inf_len, :], t_batch, z_acc_crop)

        if is_cfm:
            vel_norm   = out_crop.float().norm(dim=-1).mean().item()
            vel_step   = (out_crop * dt).float().norm(dim=-1).mean().item()
            z_hat_crop = z_t[:, :inf_len, :] - out_crop * dt
        else:
            vel_norm = vel_step = float('nan')
            z_hat_crop = recover_x0(z_t[:, :inf_len, :], out_crop, t_cur, SIGMA_MAX, PARAMETERIZATION)

        z_t_scale  = z_t[0, :N_t, :].float().norm(dim=-1).mean().item()
        zhat_scale = z_hat_crop[0, :N_t, :].float().norm(dim=-1).mean().item()
        vn = f'{vel_norm:>10.3f}' if not np.isnan(vel_norm) else f'{"—":>10}'
        vs = f'{vel_step:>10.3f}' if not np.isnan(vel_step) else f'{"—":>10}'
        print(f'{i:>4}  {t_cur.item():>5.3f}  {N_t:>5}  {z_t_scale:>10.3f}  {zhat_scale:>11.3f}  {vn}  {vs}')

        if i == N_STEPS - 1:
            break

        if is_cfm:
            z_t[:, :inf_len, :] = z_hat_crop
        else:
            z_hat_full = z_t.clone()
            z_hat_full[:, :inf_len, :] = z_hat_crop
            z_t = (1 - t_next / t_cur) * z_hat_full + (t_next / t_cur) * z_t
        z_t, _ = apply_mask(z_t, t_next.item())

step      t   N(t)   z_t_scale  z_hat_scale    vel_norm    vel_step
   0  1.000    193      41.278       35.786           —           —
   1  0.949    192      40.840       37.059           —           —
   2  0.898    191      40.479       38.411           —           —
   3  0.852    190      40.253       38.828           —           —
   4  0.801    189      40.079       39.083           —           —
   5  0.750    188      39.919       39.223           —           —
   6  0.699    187      39.789       39.321           —           —
   7  0.648    186      39.686       39.431           —           —
   8  0.602    185      39.612       39.525           —           —
   9  0.547    184      39.538       39.598           —           —
  10  0.500    183      39.497       39.673           —           —
  11  0.451    182      39.481       39.743           —           —
  12  0.400    181      39.478       39.779           —           —
  13  0.350    180      39.495       39.802     

## 5. Whisper decode — z_acc baseline vs z_nat_hat

In [ ]:
from transformers.modeling_outputs import BaseModelOutput
from src.utils.model_loader import load_baseline_whisper

whisper_model, processor = load_baseline_whisper()
whisper_model = whisper_model.to(device).eval()

def decode(z: torch.Tensor, label: str):
    enc_out = BaseModelOutput(last_hidden_state=z.unsqueeze(0).to(device))
    with torch.no_grad():
        ids = whisper_model.generate(encoder_outputs=enc_out, language='en',
                                      task='transcribe', temperature=0.0)
    text = processor.batch_decode(ids, skip_special_tokens=True)[0]
    print(f'[{label}] "{text}"')
    return text

print(f'Reference: "{PAIR["text"]}"\n')
decode(z_acc.to(torch.float32),       'z_acc (accented, no bridge)')
decode(z_nat_hat_cpu.to(torch.float32), 'z_nat_hat (bridge output)')

## 6. T_nat sensitivity — GT vs inferred

In [ ]:
# Runs inference with GT T_nat to check sensitivity (dtw_fixed: T_nat=T_l2 vs T_nat_gt).
print(f'T_nat={T_nat}  T_nat_gt={T_nat_gt}  (diff: {abs(T_nat - T_nat_gt)} frames)\n')

with torch.no_grad():
    z_hat_gt = bridge_inference(
        bridge, z_acc_bf16, T_l2=T_l2, T_nat=T_nat_gt,
        n_steps=N_STEPS, sigma_max=SIGMA_MAX, parameterization=PARAMETERIZATION,
    ).squeeze(0).float().cpu()

n = min(T_nat_gt, T_l2)
cos_hat    = torch.nn.functional.cosine_similarity(z_nat_hat_cpu[:n], z_nat[:n], dim=-1).mean()
cos_hat_gt = torch.nn.functional.cosine_similarity(z_hat_gt[:n],      z_nat[:n], dim=-1).mean()
cos_base   = torch.nn.functional.cosine_similarity(z_acc[:n],         z_nat[:n], dim=-1).mean()

print(f'cos(z_hat [T_nat={T_nat}],    z_nat): {cos_hat:.4f}')
print(f'cos(z_hat [T_nat_gt={T_nat_gt}], z_nat): {cos_hat_gt:.4f}')
print(f'cos(z_acc,                      z_nat): {cos_base:.4f}  (baseline)\n')

print(f'Reference: "{PAIR["text"]}"')
decode(z_acc.float(),      'z_acc            (no bridge)')
decode(z_nat_hat_cpu,      f'z_hat [T_nat={T_nat}]')
decode(z_hat_gt,           f'z_hat [T_nat_gt={T_nat_gt}]')
decode(z_nat.float(),      'z_nat            (ground-truth native)')

## 7. Oracle interpolation sweep — confirm on-manifold baseline

In [ ]:
import pickle
from transformers.modeling_outputs import BaseModelOutput

needs_dtw = ALIGNMENT.startswith('dtw')
if needs_dtw:
    dtw_cache_path = 'src/experiments/exp2_latent_diffusion_bridge/dtw_cache/dtw_paths.pkl'
    with open(dtw_cache_path, 'rb') as f:
        dtw_cache = pickle.load(f)
    dtw_key = (PAIR['l2_encoder_state_path'], PAIR['nat_encoder_state_path'])
    path    = dtw_cache[dtw_key]  # [P, 2] int16  path[:,0]=nat, path[:,1]=l2
    print(f'DTW path shape: {path.shape}  T_l2={T_l2}  T_nat_gt={T_nat_gt}')
else:
    path = None
    print(f'{ALIGNMENT!r}: no DTW path  T_l2={T_l2}  T_nat_gt={T_nat_gt}')

# Build alignment-specific oracle and reference target
oracle, z_nat_warped = MODE['make_oracle'](path, z_acc, z_nat, T_l2, T_nat)
# z_ref: what the model was trained to predict
#   dtw_fixed / position_fixed -> z_nat_warped [T_l2 frames]  (warped or truncated native)
#   dtw / position             -> z_nat [full 1500 frames]     (position-aligned native)
z_ref       = z_nat_warped if z_nat_warped is not None else z_nat
T_ref       = T_l2 if z_nat_warped is not None else T_nat_gt
z_ref_label = ('z_nat_warped' if ALIGNMENT == 'dtw_fixed'
               else 'z_nat_trunc' if ALIGNMENT == 'position_fixed'
               else 'z_nat')

print(f'\nReference: "{PAIR["text"]}"\n')
print(f'{"t":>5}  {"N(t)":>5}  {"cos_to_ref":>11}  {"norm":>7}  [{z_ref_label}]  transcription')
print('-' * 82)
for t in [1.0, 0.9, 0.75, 0.5, 0.25, 0.1, 0.0]:
    z_interp, N_t = oracle(t)
    n   = min(N_t, T_ref)
    cos = torch.nn.functional.cosine_similarity(z_interp[:n], z_ref[:n], dim=-1).mean().item()
    nrm = z_interp[:N_t].norm(dim=-1).mean().item()
    enc_out = BaseModelOutput(last_hidden_state=z_interp.unsqueeze(0).to(device))
    with torch.no_grad():
        ids  = whisper_model.generate(encoder_outputs=enc_out, language='en',
                                      task='transcribe', temperature=0.0)
    text = processor.batch_decode(ids, skip_special_tokens=True)[0]
    print(f'{t:>5.2f}  {N_t:>5}  {cos:>11.4f}  {nrm:>7.3f}  "{text}"')

## 8. Per-frame norm inspection — where does z_hat deviate?

In [ ]:
import matplotlib.pyplot as plt

SHOW = 250

norm_acc     = z_acc[:SHOW].norm(dim=-1).numpy()
norm_hat     = z_nat_hat_cpu[:SHOW].norm(dim=-1).numpy()
norm_ref     = z_ref[:SHOW].norm(dim=-1).numpy()
z_orc_085, _ = oracle(0.85)
norm_orc     = z_orc_085[:SHOW].norm(dim=-1).numpy()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax = axes[0]
ax.plot(norm_acc, label=f'z_acc  (T_l2={T_l2})',         alpha=0.7)
ax.plot(norm_ref, label=f'{z_ref_label}  (T={T_ref})',    alpha=0.8)
ax.plot(norm_hat, label=f'z_hat  (T_nat={T_nat})',        alpha=0.8, linestyle='--')
ax.plot(norm_orc, label=f'oracle α=0.85 (on-manifold)',   alpha=0.6, linestyle=':')
ax.axvline(T_l2,  color='steelblue', linestyle='--', alpha=0.4, label=f'T_l2={T_l2}')
ax.axvline(T_ref, color='seagreen',  linestyle='--', alpha=0.4, label=f'T_ref={T_ref}')
ax.set_ylabel('Frame L2 norm')
ax.set_title(f'Per-frame norms  [{ALIGNMENT}  {PARAMETERIZATION}  σ={SIGMA_MAX}]')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax2 = axes[1]
N = min(T_nat, T_ref, T_l2)
cos_hat_frame = torch.nn.functional.cosine_similarity(z_nat_hat_cpu[:N], z_ref[:N], dim=-1).numpy()
cos_acc_frame = torch.nn.functional.cosine_similarity(z_acc[:N],         z_ref[:N], dim=-1).numpy()
cos_orc_frame = torch.nn.functional.cosine_similarity(oracle(0.85)[0][:N], z_ref[:N], dim=-1).numpy()

ax2.plot(cos_acc_frame, label=f'cos(z_acc, {z_ref_label})',         alpha=0.8)
ax2.plot(cos_hat_frame, label=f'cos(z_hat, {z_ref_label})',         alpha=0.8, linestyle='--')
ax2.plot(cos_orc_frame, label=f'cos(oracle 0.85, {z_ref_label})',   alpha=0.6, linestyle=':')
ax2.axhline(0, color='k', linewidth=0.5)
ax2.set_ylabel(f'Per-frame cosine sim to {z_ref_label}')
ax2.set_xlabel('Frame')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Padding surgery — isolate speech vs tail as the culprit

In [ ]:
print(f'Reference: "{PAIR["text"]}"\n')

def attach_acc_silence(speech_z, T_speech, T_l2, z_acc, L=1500):
    """z[:T_speech] from speech_z, then z_acc[T_l2:] silence — tile last frame if short."""
    z = speech_z.clone()
    n_needed = L - T_speech
    n_avail  = L - T_l2
    if n_avail >= n_needed:
        z[T_speech:] = z_acc[T_l2 : T_l2 + n_needed]
    else:
        z[T_speech : T_speech + n_avail] = z_acc[T_l2:]
        z[T_speech + n_avail :]          = z_acc[-1:].expand(n_needed - n_avail, -1)
    return z

# A: z_hat speech + z_acc silence (key test — if this decodes, speech frames are fine)
z_hat_acc_sil = attach_acc_silence(z_nat_hat_cpu, T_nat_gt, T_l2, z_acc)
decode(z_hat_acc_sil, 'z_hat speech + z_acc silence @ T_nat_gt')

# B: z_acc speech + z_acc silence shifted to T_nat_gt (sanity — should decode)
z_acc_shifted = attach_acc_silence(z_acc, T_nat_gt, T_l2, z_acc)
decode(z_acc_shifted, 'z_acc speech + z_acc silence @ T_nat_gt (sanity)')

# C: on-manifold oracle at matched cos_sim — confirms on-manifold decodes at this level
n = min(T_l2, T_nat_gt)
cos_hat = torch.nn.functional.cosine_similarity(
    z_nat_hat_cpu[:n], z_nat[:n], dim=-1).mean().item()
best_alpha, best_diff = 0.5, 1.0
for a in torch.linspace(0, 1, 101):
    z_tmp = a.item() * z_acc[:n] + (1 - a.item()) * z_nat[:n]
    c = torch.nn.functional.cosine_similarity(z_tmp, z_nat[:n], dim=-1).mean().item()
    if abs(c - cos_hat) < best_diff:
        best_diff, best_alpha = abs(c - cos_hat), a.item()
z_matched, N_matched = oracle(best_alpha)
cos_matched = torch.nn.functional.cosine_similarity(
    z_matched[:n], z_nat[:n], dim=-1).mean().item()
print(f'\nMatched oracle: alpha={best_alpha:.2f} → cos={cos_matched:.4f}  (z_hat cos={cos_hat:.4f})')
decode(z_matched, f'on-manifold oracle α={best_alpha:.2f} (matched cos_sim)')

## 10. Multi-native target analysis — mode-averaging hypothesis

The training mapping pairs each L2 utterance with **all three** native speakers (CLB, SLT, RMS).
Because MSE regression minimises the mean of all targets, the model converges to predicting
their **centroid** in latent space rather than any individual speaker.

If the centroid of {CLB, SLT, RMS} sits off the Whisper encoder manifold, the bridge output
will have good cosine similarity to z_nat on paper but be undecoadable.

Key numbers to compare:
- `cos_sim(z_acc, z_nat)` — baseline gap the bridge must close
- `cos_sim(CLB, SLT/RMS) etc.` — spread of the three native targets for the same utterance
- Bridge training diagnostic cos_sim ≈ **0.59** (from history.json)

If the bridge cos_sim ≈ inter-native cos_sim, the model has learned the centroid.

In [7]:
from collections import defaultdict
import torch.nn.functional as F
import numpy as np
import random
import jiwer

TRAIN_DIR = Path(os.environ['TRAIN_DATA_DIR'])

with open('src/experiments/exp2_latent_diffusion_bridge/data/mapping_train.json') as f:
    train_map = json.load(f)

# Group entries by (l2_speaker, prompt_id) → {nat_speaker: entry}
groups = defaultdict(dict)
for u in train_map:
    key = (u['l2_speaker'], u['prompt_id'])
    groups[key][u['nat_speaker']] = u

NATIVE_SPKS = ['CLB', 'SLT', 'RMS']
PAIRS = [('CLB','SLT'), ('CLB','RMS'), ('SLT','RMS')]
triplets = {k: v for k, v in groups.items() if all(s in v for s in NATIVE_SPKS)}
print(f'Complete triplets: {len(triplets):,} / {len(groups):,} utterance keys')

sample = list(triplets.items()); random.shuffle(sample)


Complete triplets: 12,428 / 20,132 utterance keys


In [ ]:


inter_nat_sims = {(a, b): [] for a, b in PAIRS}
acc_nat_sims   = {s: [] for s in NATIVE_SPKS}
centroid_sims  = []

import random; random.seed(0)

n_ok = 0
for key, entries in sample[:300]:
    z_nats = {}
    T_nats = {}
    ok = True
    for spk in NATIVE_SPKS:
        p = TRAIN_DIR / entries[spk]['nat_encoder_state_path']
        if not p.exists(): ok = False; break
        z = torch.load(p, map_location='cpu', weights_only=False)
        if isinstance(z, dict): z = z['hidden_states']
        z_nats[spk] = z.float()
        T_nats[spk] = entries[spk]['nat_speech_end_frame']
    if not ok: continue

    e0 = entries[NATIVE_SPKS[0]]
    acc_p = TRAIN_DIR / e0['l2_encoder_state_path']
    if not acc_p.exists(): continue
    z_acc_u = torch.load(acc_p, map_location='cpu', weights_only=False)
    if isinstance(z_acc_u, dict): z_acc_u = z_acc_u['hidden_states']
    z_acc_u = z_acc_u.float()
    T_l2_u = e0['l2_speech_end_frame']

    # Inter-native: compare each pair over min of their two speech lengths
    for (a, b) in PAIRS:
        T_ab = min(T_nats[a], T_nats[b])
        if T_ab < 5: continue
        inter_nat_sims[(a, b)].append(
            F.cosine_similarity(z_nats[a][:T_ab], z_nats[b][:T_ab], dim=-1).mean().item())

    # z_acc vs z_nat: over min of L2 speech and each native speech length
    for spk in NATIVE_SPKS:
        T_an = min(T_l2_u, T_nats[spk])
        if T_an < 5: continue
        acc_nat_sims[spk].append(
            F.cosine_similarity(z_acc_u[:T_an], z_nats[spk][:T_an], dim=-1).mean().item())

    # Centroid: average [0:T_max] across all 3 speakers (speech frames only)
    T_max = max(T_nats.values())
    centroid_speech = torch.stack([z_nats[spk][:T_max] for spk in NATIVE_SPKS]).mean(0)
    for spk in NATIVE_SPKS:
        T_cmp = min(T_nats[spk], T_max)
        centroid_sims.append(
            F.cosine_similarity(centroid_speech[:T_cmp], z_nats[spk][:T_cmp], dim=-1).mean().item())

    n_ok += 1

print(f'Processed {n_ok} utterances\n')

all_inter   = [s for v in inter_nat_sims.values() for s in v]
all_acc_nat = [s for v in acc_nat_sims.values() for s in v]

print('── Inter-native cosine sim (speech frames only) ──')
for (a, b), sims in inter_nat_sims.items():
    print(f'  {a}↔{b}: {np.mean(sims):.4f} ± {np.std(sims):.4f}')
print(f'  overall:  {np.mean(all_inter):.4f} ± {np.std(all_inter):.4f}')

print('\n── z_acc vs z_nat cosine sim (speech frames only) ──')
for spk, sims in acc_nat_sims.items():
    print(f'  z_acc ↔ {spk}: {np.mean(sims):.4f} ± {np.std(sims):.4f}')
print(f'  z_acc ↔ all:  {np.mean(all_acc_nat):.4f} ± {np.std(all_acc_nat):.4f}')

print(f'\n── Centroid cos_sim to each z_nat (speech frames only) ──')
print(f'  {np.mean(centroid_sims):.4f} ± {np.std(centroid_sims):.4f}')

bridge_diagnostic_sim = 0.590
print(f'\n── Bridge training diagnostic cos_sim (from history.json) ──')
print(f'  {bridge_diagnostic_sim:.4f}  (flat across all 15 epochs)')
print(f'\n→ centroid sim = {np.mean(centroid_sims):.4f},  bridge sim = {bridge_diagnostic_sim:.4f}')
print(f'  gap = {abs(np.mean(centroid_sims) - bridge_diagnostic_sim):.4f}')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: boxplot of the three sim distributions
ax = axes[0]
data = [all_acc_nat, all_inter, centroid_sims]
labels = ['z_acc ↔ z_nat\n(baseline gap)', 'inter-native\n(CLB/SLT/RMS)', 'centroid ↔ z_nat\n(mode-avg target)']
bp = ax.boxplot(data, labels=labels, patch_artist=True, medianprops=dict(color='k', linewidth=2))
colors = ['#4e79a7', '#f28e2b', '#e15759']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.axhline(bridge_diagnostic_sim, color='purple', linestyle='--', linewidth=1.5,
           label=f'bridge diagnostic ({bridge_diagnostic_sim:.3f})')
ax.set_ylabel('Cosine similarity')
ax.set_title('Latent space distances (training data, n=300 utterances)')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Right: per-pair inter-native breakdown
ax2 = axes[1]
pair_labels = [f'{a}↔{b}' for a, b in PAIRS]
pair_means  = [np.mean(inter_nat_sims[(a, b)]) for a, b in PAIRS]
pair_stds   = [np.std(inter_nat_sims[(a, b)])  for a, b in PAIRS]
ax2.bar(pair_labels, pair_means, yerr=pair_stds, capsize=5,
        color='#f28e2b', alpha=0.7, edgecolor='k')
ax2.axhline(bridge_diagnostic_sim, color='purple', linestyle='--', linewidth=1.5,
            label=f'bridge sim ({bridge_diagnostic_sim:.3f})')
ax2.axhline(np.mean(all_acc_nat), color='#4e79a7', linestyle=':', linewidth=1.5,
            label=f'z_acc↔z_nat ({np.mean(all_acc_nat):.3f})')
ax2.set_ylabel('Cosine similarity')
ax2.set_title('Inter-native speaker distance per pair')
ax2.set_ylim(0.3, 0.9)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Mode-averaging hypothesis: bridge cos_sim ≈ centroid cos_sim', fontsize=11)
plt.tight_layout()
plt.savefig('notebooks/centroid_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/centroid_analysis.png')

## 11. Centroid WER — is the mode-averaged latent decodable?

Also checks each individual native speaker for reference.
If centroid decodes well → it's on-manifold and would be a better training target than individual z_nats.
If not → mode averaging is off-manifold regardless.

In [ ]:
import jiwer
from transformers.modeling_outputs import BaseModelOutput
from tqdm import tqdm

def whisper_decode(z_full: torch.Tensor) -> str:
    enc_out = BaseModelOutput(last_hidden_state=z_full.unsqueeze(0).to(device))
    with torch.no_grad():
        ids = whisper_model.generate(encoder_outputs=enc_out, language='en',
                                     task='transcribe', temperature=0.0)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].lower().strip()

def corpus_wer(refs, hyps):
    return float(jiwer.process_words(refs, hyps).wer)

# Sample utterances that have all three native speakers
n_eval = min(100, len(sample))
rows = {'CLB': [], 'SLT': [], 'RMS': [], 'centroid': []}
refs = []

print(f'Decoding {n_eval} utterances - individual speakers + centroid ...')
n_done = 0
for key, entries in tqdm(sample[:n_eval * 3], desc='Evaluating'):
    if n_done >= n_eval:
        break

    z_nats_u = {}
    ok = True
    for spk in NATIVE_SPKS:
        p = TRAIN_DIR / entries[spk]['nat_encoder_state_path']
        if not p.exists(): ok = False; break
        z = torch.load(p, map_location='cpu', weights_only=False)
        if isinstance(z, dict): z = z['hidden_states']
        z_nats_u[spk] = z.float()
    if not ok:
        continue

    ref = entries[NATIVE_SPKS[0]]['text'].lower().strip()
    import re; ref = re.sub(r'[^\w\s]', '', ref)

    T_nats_u = {spk: entries[spk]['nat_speech_end_frame'] for spk in NATIVE_SPKS}
    T_max_spk = max(T_nats_u, key=T_nats_u.get)
    T_max_u   = T_nats_u[T_max_spk]
    # Average speech frames [0:T_max], attach silence from longest speaker
    centroid_speech = torch.stack([z_nats_u[spk][:T_max_u] for spk in NATIVE_SPKS]).mean(0)
    centroid_full   = torch.cat([centroid_speech, z_nats_u[T_max_spk][T_max_u:]], dim=0)

    refs.append(ref)
    for spk in NATIVE_SPKS:
        rows[spk].append(whisper_decode(z_nats_u[spk]))
    rows['centroid'].append(whisper_decode(centroid_full))
    n_done += 1

print(f'\nEvaluated {n_done} utterances\n')
print(f'{"Source":>10}  {"WER":>7}  note')
print('-' * 45)
for label in ['CLB', 'SLT', 'RMS', 'centroid']:
    wer = corpus_wer(refs, rows[label])
    note = '<- centroid of all 3' if label == 'centroid' else ''
    print(f'{label:>10}  {wer:>7.3f}  {note}')

print('\nSample predictions (first 5):')
for i in range(min(5, n_done)):
    print(f'  ref:      {refs[i]}')
    for label in ['CLB', 'SLT', 'RMS', 'centroid']:
        print(f'  {label:>8}: {rows[label][i]}')
    print()


In [ ]:
from src.experiments.exp2_latent_diffusion_bridge.diffusion import bridge_inference

bridge_rows = []
n_bridge = 0

for key, entries in sample[:n_eval * 3]:
    if n_bridge >= n_done:
        break
    e0 = entries[NATIVE_SPKS[0]]
    acc_p = TRAIN_DIR / e0['l2_encoder_state_path']
    if not acc_p.exists(): continue
    if not all((TRAIN_DIR / entries[spk]['nat_encoder_state_path']).exists() for spk in NATIVE_SPKS): continue
    z_acc_u = torch.load(acc_p, map_location='cpu', weights_only=False)
    if isinstance(z_acc_u, dict): z_acc_u = z_acc_u['hidden_states']
    T_l2_u  = e0['l2_speech_end_frame']
    T_nat_u = e0['nat_speech_end_frame']
    T_nat_inf_u = T_l2_u if not MODE['uses_predictor'] else T_nat_u
    z_acc_bf = z_acc_u.float().to(device=device, dtype=torch.bfloat16).unsqueeze(0)
    with torch.no_grad():
        z_hat = bridge_inference(
            bridge, z_acc_bf, T_l2=T_l2_u, T_nat=T_nat_inf_u,
            n_steps=N_STEPS, sigma_max=SIGMA_MAX,
            parameterization=PARAMETERIZATION,
        ).squeeze(0).float().cpu()
    bridge_rows.append(whisper_decode(z_hat))
    n_bridge += 1

print(f'Bridge decoded {n_bridge} utterances\n')
print(f'{"Source":>10}  {"WER":>7}')
print('-' * 30)
for label in ['CLB', 'SLT', 'RMS', 'centroid']:
    print(f'{label:>10}  {corpus_wer(refs[:n_bridge], rows[label][:n_bridge]):>7.3f}')
print(f'{"bridge":>10}  {corpus_wer(refs[:n_bridge], bridge_rows):>7.3f}')

print('\nSample predictions (first 10):')
for i in range(min(10, n_bridge)):
    print(f'  ref:      {refs[i]}')
    for label in ['CLB', 'SLT', 'RMS']:
        print(f'  {label:>8}: {rows[label][i]}')
    print(f'  {"centroid":>8}: {rows["centroid"][i]}')
    print(f'  {"bridge":>8}: {bridge_rows[i]}')
    print()

In [ ]:
import re

def norm_text(s):
    s = s.lower().strip()
    return re.sub(r'[^\w\s]', '', s)

def utt_wer(ref, hyp):
    if not ref: return 0.0
    return float(jiwer.process_words([norm_text(ref)], [norm_text(hyp)]).wer)

scored = sorted(
    [(utt_wer(refs[i], bridge_rows[i]), refs[i], bridge_rows[i], rows['centroid'][i])
     for i in range(n_bridge)],
    key=lambda x: -x[0]
)

print('=== Worst 10 bridge outputs ===\n')
for wer, ref, bridge_pred, centroid_pred in scored[:10]:
    print(f'  WER={wer:.2f}')
    print(f'  ref:      {ref}')
    print(f'  centroid: {centroid_pred}')
    print(f'  bridge:   {bridge_pred}')
    print()

print('=== Best 10 bridge outputs ===\n')
for wer, ref, bridge_pred, centroid_pred in scored[-10:]:
    print(f'  WER={wer:.2f}')
    print(f'  ref:      {ref}')
    print(f'  centroid: {centroid_pred}')
    print(f'  bridge:   {bridge_pred}')
    print()

n_perfect = sum(1 for w, *_ in scored if w == 0.0)
n_total = len(scored)
print(f'Perfect (WER=0): {n_perfect}/{n_total}  ({100*n_perfect/n_total:.0f}%)')
print(f'WER > 1.0:       {sum(1 for w,*_ in scored if w > 1.0)}/{n_total}')


## 12. ODE Trajectory Diagnostics — where does the bridge actually move?

At each ODE step (t: 1→0) track:
- **cos(z_t, z_acc)**: movement away from source
- **cos(z_t, z_nat)**: movement toward target (should rise monotonically if bridge is working)
- **cos(x̃₀, z_nat)**: model's x0 prediction quality at each step
- **direction alignment**: cos(ODE step vector, ideal direction to z_nat) — are we stepping the right way?

Then compare trajectories across good (WER≈0) vs bad (WER>>1) utterances, and project into 2D via PCA.

In [8]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

# z_ref / z_nat_warped set in oracle cell (section 7)
# dtw_fixed: z_ref = z_nat_warped [T_l2 frames]; dtw: z_ref = z_nat [1500 frames]
T_ref_len = T_l2 if z_nat_warped is not None else T_nat_gt
T_ref_sp  = min(T_l2, T_ref_len)

def speech_cos(a, b, T):
    n = min(T, a.shape[0], b.shape[0])
    return F.cosine_similarity(a[:n].float(), b[:n].float(), dim=-1).mean().item()

B, L, D    = z_acc_bf16.shape
dtype      = z_acc_bf16.dtype
sil_s      = z_acc_bf16[:, T_l2:T_l2 + 1, :].clone()
inf_len    = max(T_l2, T_nat) + 1
z_acc_crop = z_acc_bf16[:, :inf_len, :]
t_sched    = torch.linspace(1.0, 0.0, N_STEPS + 1, device=device, dtype=dtype)

def apply_mask(z, t_val):
    N_t = MODE['n_frames'](t_val, T_l2, T_nat)
    z[:, N_t:, :] = sil_s.expand(B, L - N_t, D)
    return z, N_t

cos_zt_acc, cos_zt_ref, cos_pred_ref, dir_align, n_t_vals, t_vals = [], [], [], [], [], []

z_t = z_acc_bf16.clone()
z_t, _ = apply_mask(z_t, 1.0)

bridge.eval()
with torch.no_grad():
    for i in range(N_STEPS):
        t_cur  = t_sched[i]
        t_next = t_sched[i + 1]
        dt     = (t_cur - t_next).item()
        t_batch = torch.full((B,), t_cur, device=device, dtype=dtype)

        out_crop = bridge(z_t[:, :inf_len, :], t_batch, z_acc_crop)

        if PARAMETERIZATION == 'cfm':
            z_hat_crop = z_t[:, :inf_len, :] - out_crop * dt
            step_v     = (-out_crop[0].float().cpu()[:T_ref_sp]).reshape(-1)
        else:
            z_hat_crop = recover_x0(z_t[:, :inf_len, :], out_crop, t_cur, SIGMA_MAX, PARAMETERIZATION)
            zt_cpu_tmp = z_t.squeeze(0).float().cpu()
            step_v     = (z_hat_crop[0].float().cpu()[:T_ref_sp] - zt_cpu_tmp[:T_ref_sp]).reshape(-1)

        z_hat_full = z_t.clone()
        z_hat_full[:, :inf_len, :] = z_hat_crop

        zt_cpu   = z_t.squeeze(0).float().cpu()
        pred_cpu = z_hat_full.squeeze(0).float().cpu()
        ideal_v  = (z_ref[:T_ref_sp] - zt_cpu[:T_ref_sp]).reshape(-1)

        cos_zt_acc.append(speech_cos(zt_cpu,  z_acc, T_ref_sp))
        cos_zt_ref.append(speech_cos(zt_cpu,  z_ref, T_ref_sp))
        cos_pred_ref.append(speech_cos(pred_cpu, z_ref, T_ref_sp))
        dir_align.append(
            F.cosine_similarity(step_v.unsqueeze(0), ideal_v.unsqueeze(0)).item()
            if step_v.norm() > 1e-6 and ideal_v.norm() > 1e-6 else float('nan')
        )
        t_vals.append(t_cur.item())
        n_t_vals.append(MODE['n_frames'](t_cur.item(), T_l2, T_nat))

        if i == N_STEPS - 1:
            break

        if PARAMETERIZATION == 'cfm':
            z_t[:, :inf_len, :] = z_hat_crop
        else:
            z_t = (1 - t_next / t_cur) * z_hat_full + (t_next / t_cur) * z_t
        z_t, _ = apply_mask(z_t, t_next.item())

baseline_sim = speech_cos(z_acc, z_ref, T_ref_sp)
pred_label   = f'cos(z_t−v·dt, {z_ref_label})' if PARAMETERIZATION == 'cfm' else f'cos(x̃₀, {z_ref_label})'
step_label   = 'cos(−v, toward ref)' if PARAMETERIZATION == 'cfm' else 'cos(step, toward ref)'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(t_vals, cos_zt_acc,  label='cos(z_t, z_acc)',          color='steelblue', lw=2)
ax.plot(t_vals, cos_zt_ref,  label=f'cos(z_t, {z_ref_label})', color='seagreen',  lw=2)
ax.plot(t_vals, cos_pred_ref, label=pred_label,                 color='tomato',    lw=2, linestyle='--')
ax.axhline(baseline_sim, color='gray', linestyle=':', lw=1, label=f'baseline {baseline_sim:.3f}')
ax.invert_xaxis()
ax.set_xlabel('t  (1=accented → 0=native)')
ax.set_ylabel('Cosine similarity')
ax.set_title('Trajectory: cosine similarities')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(t_vals, dir_align, color='purple', lw=2)
ax2.axhline(0, color='k', lw=0.5, linestyle='--')
ax2.axhline(1, color='seagreen', lw=0.5, linestyle=':')
ax2.invert_xaxis()
ax2.set_xlabel('t')
ax2.set_ylabel(step_label)
ax2.set_title('Direction alignment\n(+1=toward ref, 0=perp, −1=wrong way)')
ax2.set_ylim(-1.1, 1.1); ax2.grid(alpha=0.3)

ax3 = axes[2]
ax3.plot(t_vals, n_t_vals, color='orange', lw=2)
ax3.axhline(T_l2,     color='steelblue', linestyle=':', lw=1, label=f'T_l2={T_l2}')
ax3.axhline(T_nat_gt, color='seagreen',  linestyle=':', lw=1, label=f'T_nat_gt={T_nat_gt}')
ax3.invert_xaxis()
ax3.set_xlabel('t')
ax3.set_ylabel('N(t)  active frames')
ax3.set_title('N(t) schedule')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

plt.suptitle(
    f'{PAIR["l2_speaker"]} {PAIR["l2_utterance_id"]}  |  '
    f'T_l2={T_l2}  T_nat_gt={T_nat_gt}  T_nat={T_nat}  |  '
    f'alignment={ALIGNMENT}  param={PARAMETERIZATION}  \u03c3={SIGMA_MAX}',
    fontsize=9)
plt.tight_layout()
plt.savefig('notebooks/trajectory_single.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/trajectory_single.png')

NameError: name 'z_nat_warped' is not defined

In [ ]:
import matplotlib.cm as cm
import re

N_TRACE = 25

def norm_text(s):
    return re.sub(r'[^\w\s]', '', s.lower().strip())

traj_records = []

bridge.eval()
for key, entries in sample[:N_TRACE * 5]:
    if len(traj_records) >= N_TRACE:
        break
    e0    = entries[NATIVE_SPKS[0]]
    acc_p = TRAIN_DIR / e0['l2_encoder_state_path']
    nat_p = TRAIN_DIR / entries['CLB']['nat_encoder_state_path']
    if not acc_p.exists() or not nat_p.exists():
        continue

    z_a = torch.load(acc_p, map_location='cpu', weights_only=False)
    z_n = torch.load(nat_p,  map_location='cpu', weights_only=False)
    z_a = (z_a['hidden_states'] if isinstance(z_a, dict) else z_a).float()
    z_n = (z_n['hidden_states'] if isinstance(z_n, dict) else z_n).float()

    Tl2      = e0['l2_speech_end_frame']
    Tnat_gt  = entries['CLB']['nat_speech_end_frame']
    Tnat_inf = Tl2 if not MODE['uses_predictor'] else Tnat_gt
    Tref     = min(Tl2, Tnat_gt)
    if Tref < 20:
        continue

    z_a_dev  = z_a.to(device=device, dtype=torch.bfloat16).unsqueeze(0)
    il       = max(Tl2, Tnat_inf) + 1
    z_a_crop = z_a_dev[:, :il, :]
    t_sch    = torch.linspace(1.0, 0.0, N_STEPS + 1, device=device, dtype=torch.bfloat16)
    sil_u    = z_a_dev[:, Tl2:Tl2 + 1, :].clone()
    D_u      = z_a_dev.shape[-1]

    def _mask(z, tv):
        Nt = MODE['n_frames'](tv, Tl2, Tnat_inf)
        z[:, Nt:, :] = sil_u.expand(1, 1500 - Nt, D_u)
        return z

    cos_zt = []
    cos_pr = []
    z_t_u  = z_a_dev.clone()
    z_t_u  = _mask(z_t_u, 1.0)

    with torch.no_grad():
        for i in range(N_STEPS):
            tc = t_sch[i]; tn = t_sch[i + 1]
            tb = torch.full((1,), tc, device=device, dtype=torch.bfloat16)

            out        = bridge(z_t_u[:, :il, :], tb, z_a_crop)
            z_hat_crop = recover_x0(z_t_u[:, :il, :], out, tc, SIGMA_MAX, PARAMETERIZATION)
            zp         = z_t_u.clone()
            zp[:, :il, :] = z_hat_crop

            zt_c = z_t_u.squeeze(0).float().cpu()
            zp_c = zp.squeeze(0).float().cpu()
            cos_zt.append(speech_cos(zt_c, z_n, Tref))
            cos_pr.append(speech_cos(zp_c, z_n, Tref))

            if i == N_STEPS - 1:
                break
            z_t_u = (1 - tn / tc) * zp + (tn / tc) * z_t_u
            z_t_u = _mask(z_t_u, tn.item())

    from transformers.modeling_outputs import BaseModelOutput
    enc = BaseModelOutput(last_hidden_state=zp.float())
    with torch.no_grad():
        ids  = whisper_model.generate(encoder_outputs=enc, language='en',
                                      task='transcribe', temperature=0.0)
    pred = processor.batch_decode(ids, skip_special_tokens=True)[0]
    ref  = norm_text(e0['text'])
    wer  = float(jiwer.process_words([ref], [norm_text(pred)]).wer) if ref else 1.0

    traj_records.append({'cos_zt_nat': cos_zt, 'cos_pred_nat': cos_pr,
                         'wer': wer, 'ref': ref, 'pred': norm_text(pred)})

print(f'Traced {len(traj_records)} utterances')
wers = [r['wer'] for r in traj_records]
print(f'WER: min={min(wers):.3f}  median={sorted(wers)[len(wers)//2]:.3f}  max={max(wers):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
t_ax = [t_sched[i].item() for i in range(N_STEPS)]

for rec in traj_records:
    c = cm.RdYlGn_r(min(rec['wer'] / 2.0, 1.0))
    axes[0].plot(t_ax, rec['cos_zt_nat'],   color=c, alpha=0.7, lw=1.2)
    axes[1].plot(t_ax, rec['cos_pred_nat'], color=c, alpha=0.7, lw=1.2)

for ax, title in zip(axes, ['cos(z_t, z_nat) — trajectory', 'cos(x̃₀, z_nat) — prediction']):
    ax.invert_xaxis()
    ax.set_xlabel('t  (1=accented → 0=native)')
    ax.set_ylabel('Cosine similarity')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.axhline(baseline_sim, color='gray', linestyle=':', lw=1, label='baseline')
    ax.legend(fontsize=8)

sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=plt.Normalize(0, 2))
sm.set_array([])
plt.colorbar(sm, ax=axes, label='Final bridge WER (green=0, red\u22652)', shrink=0.8)
plt.suptitle(
    f'ODE trajectories: {len(traj_records)} utterances  |  {ALIGNMENT}  {PARAMETERIZATION}  \u03c3={SIGMA_MAX}',
    fontsize=11)
plt.tight_layout()
plt.savefig('notebooks/trajectory_multi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/trajectory_multi.png')

In [9]:
from sklearn.decomposition import PCA
import numpy as np

# T_ref_sp set in s12_single; reuse as comparison window for PCA
T_ref = T_ref_sp

B, L, D    = z_acc_bf16.shape
dtype      = z_acc_bf16.dtype
inf_len    = max(T_l2, T_nat) + 1
z_acc_crop = z_acc_bf16[:, :inf_len, :]
t_sched_pca = torch.linspace(1.0, 0.0, 101, device=device, dtype=dtype)

zt_means   = []
pred_means = []

z_t = z_acc_bf16.clone()
z_t, _ = apply_mask(z_t, 1.0)
zt_means.append(z_t.squeeze(0).float().cpu()[:T_ref].mean(0).numpy())

bridge.eval()
with torch.no_grad():
    for i in range(100):
        t_cur  = t_sched_pca[i]
        t_next = t_sched_pca[i + 1]
        dt     = (t_cur - t_next).item()
        t_batch = torch.full((B,), t_cur, device=device, dtype=dtype)

        out_crop = bridge(z_t[:, :inf_len, :], t_batch, z_acc_crop)

        if PARAMETERIZATION == 'cfm':
            z_hat_crop = z_t[:, :inf_len, :] - out_crop * dt
            z_hat_full = z_t.clone(); z_hat_full[:, :inf_len, :] = z_hat_crop
        else:
            z_hat_crop = recover_x0(z_t[:, :inf_len, :], out_crop, t_cur, SIGMA_MAX, PARAMETERIZATION)
            z_hat_full = z_t.clone(); z_hat_full[:, :inf_len, :] = z_hat_crop

        pred_means.append(z_hat_full.squeeze(0).float().cpu()[:T_ref].mean(0).numpy())

        if i == 99:
            break

        if PARAMETERIZATION == 'cfm':
            z_t[:, :inf_len, :] = z_hat_crop
        else:
            z_t = (1 - t_next / t_cur) * z_hat_full + (t_next / t_cur) * z_t
        z_t, _ = apply_mask(z_t, t_next.item())
        zt_means.append(z_t.squeeze(0).float().cpu()[:T_ref].mean(0).numpy())

# Oracle interpolation points
oracle_ts    = [1.0, 0.75, 0.5, 0.25, 0.0]
oracle_means = [oracle(t)[0][:min(oracle(t)[1], T_ref)].mean(0).numpy() for t in oracle_ts]

z_acc_mean = z_acc[:T_ref].mean(0).numpy()
z_ref_mean = z_ref[:T_ref].mean(0).numpy()

all_vecs = np.stack(zt_means + pred_means + oracle_means + [z_acc_mean, z_ref_mean])
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_vecs)
print(f'PCA explained variance: {pca.explained_variance_ratio_[:2].sum():.3f}')

n_zt, n_pred, n_orac = len(zt_means), len(pred_means), len(oracle_means)
zt_2d     = all_2d[:n_zt]
pred_2d   = all_2d[n_zt:n_zt + n_pred]
oracle_2d = all_2d[n_zt + n_pred:n_zt + n_pred + n_orac]
acc_2d    = all_2d[-2]
ref_2d    = all_2d[-1]

pred_line_label = 'z_t−v·dt (next step)' if PARAMETERIZATION == 'cfm' else 'x̃₀ prediction at each step'

fig, ax = plt.subplots(figsize=(9, 7))

ax.plot(zt_2d[:, 0], zt_2d[:, 1], 'o-', color='steelblue', lw=2, ms=5,
        label='z_t (bridge ODE)')
ax.annotate('t=1 (z_acc)', zt_2d[0],  fontsize=10, ha='left')
ax.annotate('t=0 (final)', zt_2d[-1], fontsize=10, ha='right')

ax.plot(pred_2d[:, 0], pred_2d[:, 1], 'x-', color='tomato', lw=1.2, ms=6,
        label=pred_line_label, zorder=5, alpha=0.8)

ax.plot(oracle_2d[:, 0], oracle_2d[:, 1], 's--', color='seagreen', lw=1.5, ms=6,
        label=f'Oracle {ALIGNMENT} (on-manifold)')
for i, tv in enumerate(oracle_ts):
    ax.annotate(f't={tv}', oracle_2d[i], fontsize=7, color='seagreen')

ax.scatter(*acc_2d, marker='*', s=200, color='navy', zorder=6, label='z_acc')
ax.scatter(*ref_2d, marker='*', s=200, color='gold',      zorder=6, label=f'{z_ref_label} (GT ref)')
# ax.annotate('z_acc',       acc_2d, fontsize=14, fontweight='bold', color='steelblue')
# ax.annotate(z_ref_label,   ref_2d, fontsize=14, fontweight='bold', color='darkgoldenrod')
ax.plot([acc_2d[0], ref_2d[0]], [acc_2d[1], ref_2d[1]], ':', color='gray', lw=1,
        label=f'z_acc → {z_ref_label}')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} var)')
ax.set_title(
    f'PCA projection of ODE trajectory\n'
    f'{PAIR["l2_speaker"]} {PAIR["l2_utterance_id"]}  |  '
    f'\u03c3={SIGMA_MAX}  |  {PARAMETERIZATION}  |  {ALIGNMENT}')
ax.legend(fontsize=12); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/trajectory_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/trajectory_pca.png')

NameError: name 'T_ref_sp' is not defined

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from src.experiments.exp2_latent_diffusion_bridge.eval import load_bridge_model

COMPARE_MODELS = [
    # ('models/bridge_dtw_cfm_prewarp',   'DTW CFM L2 Timeline', 'orangered'),
    # ('models/bridge_dtw_v2baseline',    'DTW x0 (v2)',         'steelblue'),
    # ('models/bridge_dtw_directionloss', 'DTW x0 +dir',         'mediumpurple'),
    ('models/bridge_dtw_fixed_eps_0.5',    'DTW (fixed) eps (0.5)',   'seagreen'),
    ('models/bridge_dtw_eps_0.5',          'DTW eps (0.5)',           'navy'),
    ('models/bridge_position_eps_0.5',     'Position eps (0.5)',      'red'),
    ('models/bridge_dtw_fixed_x0_0.5',     'DTW (fixed) z0 (0.5)',    'mediumpurple'),
]
N_ODE_STEPS = 100


def run_trajectory(model, parameterization, sigma_mx, z_acc_in, z_acc_crop_in,
                   inf_len, n_steps, T_ref_win, apply_mask_fn):
    """Collect mean-speech z_t at each ODE step."""
    B, L, D  = z_acc_in.shape
    dtype    = z_acc_in.dtype
    t_sch    = torch.linspace(1.0, 0.0, n_steps + 1, device=device, dtype=dtype)

    z_t = z_acc_in.clone()
    z_t, _ = apply_mask_fn(z_t, 1.0)
    means   = [z_t.squeeze(0).float().cpu()[:T_ref_win].mean(0).numpy()]

    model.eval()
    with torch.no_grad():
        for i in range(n_steps):
            t_cur  = t_sch[i]
            t_next = t_sch[i + 1]
            dt     = (t_cur - t_next).item()
            tb     = torch.full((B,), t_cur, device=device, dtype=dtype)
            out    = model(z_t[:, :inf_len, :], tb, z_acc_crop_in)

            if parameterization == 'cfm':
                z_hat_crop = z_t[:, :inf_len, :] - out * dt
            else:
                z_hat_crop = recover_x0(z_t[:, :inf_len, :], out, t_cur, sigma_mx, parameterization)
            z_hat_full = z_t.clone(); z_hat_full[:, :inf_len, :] = z_hat_crop

            if i == n_steps - 1:
                break

            if parameterization == 'cfm':
                z_t[:, :inf_len, :] = z_hat_crop
            else:
                z_t = (1 - t_next / t_cur) * z_hat_full + (t_next / t_cur) * z_t
            z_t, _ = apply_mask_fn(z_t, t_next.item())
            means.append(z_t.squeeze(0).float().cpu()[:T_ref_win].mean(0).numpy())

    return means


# Oracle reference points (use current PAIR's oracle)
oracle_ts    = [1.0, 0.75, 0.5, 0.25, 0.0]
oracle_means = [oracle(t)[0][:min(oracle(t)[1], T_ref_sp)].mean(0).numpy() for t in oracle_ts]

z_acc_mean = z_acc[:T_ref_sp].mean(0).numpy()
z_ref_mean = z_ref[:T_ref_sp].mean(0).numpy()

# Run each model
all_traj = {}
for ckpt_dir, label, color in COMPARE_MODELS:
    ckpt_path = Path(ckpt_dir) / 'checkpoint_best.pt'
    cfg_path  = Path(ckpt_dir) / 'config.json'
    if not ckpt_path.exists():
        print(f'[skip] {label}: no checkpoint')
        continue
    cfg          = json.loads(cfg_path.read_text())
    param        = cfg['parameterization']
    sigma_mx     = float(cfg['sigma_max'])
    model_align  = cfg.get('alignment', ALIGNMENT)
    model_mode   = ALIGNMENT_MODES.get(model_align, MODE)
    Tnat_m       = T_l2 if not model_mode['uses_predictor'] else T_nat
    inf_len_m    = max(T_l2, Tnat_m) + 1
    sil_m        = z_acc_bf16[:, T_l2:T_l2 + 1, :].clone()
    L_m          = z_acc_bf16.shape[1]
    D_m          = z_acc_bf16.shape[2]

    def _make_mask(mode, tl2, tnat_inf, sil, L, D):
        def _mask(z, tv):
            Nt = mode['n_frames'](tv, tl2, tnat_inf)
            z[:, Nt:, :] = sil.expand(1, L - Nt, D)
            return z, Nt
        return _mask

    m     = load_bridge_model(str(ckpt_path), device)
    _mask = _make_mask(model_mode, T_l2, Tnat_m, sil_m, L_m, D_m)
    traj  = run_trajectory(m, param, sigma_mx, z_acc_bf16.clone(),
                           z_acc_bf16[:, :inf_len_m, :], inf_len_m,
                           N_ODE_STEPS, T_ref_sp, _mask)
    all_traj[label] = traj
    print(f'  [{label}] align={model_align}  {param}  sigma={sigma_mx}  {len(traj)} steps')

# Shared PCA
all_vecs = [v for traj in all_traj.values() for v in traj] + oracle_means + [z_acc_mean, z_ref_mean]
pca      = PCA(n_components=2)
all_2d   = pca.fit_transform(np.stack(all_vecs))
print(f'PCA variance: {pca.explained_variance_ratio_[:2].sum():.3f}')

idx, traj_2d = 0, {}
for label, traj in all_traj.items():
    traj_2d[label] = all_2d[idx: idx + len(traj)]; idx += len(traj)
oracle_2d = all_2d[idx: idx + len(oracle_means)]
acc_2d    = all_2d[-2]
ref_2d    = all_2d[-1]

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(oracle_2d[:, 0], oracle_2d[:, 1], 's--', color='seagreen', lw=1.8, ms=7,
        label=f'Oracle {ALIGNMENT} (on-manifold)', zorder=3)
for i, tv in enumerate(oracle_ts):
    ax.annotate(f't={tv}', oracle_2d[i], fontsize=7, color='seagreen')

color_map = {label: color for _, label, color in COMPARE_MODELS}
for label, pts in traj_2d.items():
    c = color_map.get(label, 'gray')
    ax.plot(pts[:, 0], pts[:, 1], '-', color=c, lw=2, label=label, alpha=0.85)
    ax.scatter(*pts[0],  marker='o', s=55, color=c, zorder=5)
    ax.scatter(*pts[-1], marker='X', s=75, color=c, zorder=5)

ax.scatter(*acc_2d, marker='*', s=250, color='navy', zorder=6, label='z_acc')
ax.scatter(*ref_2d, marker='*', s=250, color='gold', zorder=6, label=f'{z_ref_label} (GT ref)')
ax.annotate('z_acc',     acc_2d, fontsize=9, fontweight='bold', color='navy',
            xytext=(4,4), textcoords='offset points')
ax.annotate(z_ref_label, ref_2d, fontsize=9, fontweight='bold', color='darkgoldenrod',
            xytext=(4,4), textcoords='offset points')
ax.plot([acc_2d[0], ref_2d[0]], [acc_2d[1], ref_2d[1]], ':', color='gray',
        lw=1, label=f'z_acc→{z_ref_label}')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title(
    f'Multi-model ODE trajectories (shared PCA)\n'
    f'{PAIR["l2_speaker"]} {PAIR["l2_utterance_id"]}  '
    f'T_l2={T_l2}  T_nat_gt={T_nat_gt}  T_nat={T_nat}')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/trajectory_pca_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/trajectory_pca_compare.png')

In [10]:
import json
import torch
import torch.nn.functional as F
from pathlib import Path

dev_mapping = json.load(open('src/experiments/exp2_latent_diffusion_bridge/data/mapping_dev_v2.json'))
other_entry = next(e for e in dev_mapping if e['l2_utterance_id'] != PAIR['l2_utterance_id'])
other_path  = Path(os.environ['TRAIN_DATA_DIR']) / other_entry['l2_encoder_state_path']
z_acc_other_raw  = torch.load(other_path, map_location='cpu', weights_only=False)
z_acc_other_bf16 = z_acc_other_raw['hidden_states'].to(device=device, dtype=torch.bfloat16).unsqueeze(0)
print(f"Other utterance: {other_entry['l2_utterance_id']}  T_l2={other_entry['l2_speech_end_frame']}")
print(f"This utterance:  {PAIR['l2_utterance_id']}  T_l2={T_l2}")

inf_len    = max(T_l2, T_nat) + 1
z_acc_crop = z_acc_bf16[:, :inf_len, :]

def run_inference_with_z_t_init(z_t_init):
    """Run bridge ODE with custom initial z_t; z_acc conditioning fixed."""
    z_t = z_t_init.clone().to(device=device, dtype=torch.bfloat16)
    if z_t.dim() == 2: z_t = z_t.unsqueeze(0)
    t_schedule = torch.linspace(1.0, 0.0, N_STEPS + 1, device=device, dtype=torch.bfloat16)
    bridge.eval()
    with torch.no_grad():
        for i in range(N_STEPS):
            t_cur  = t_schedule[i]
            t_next = t_schedule[i + 1]
            dt     = (t_cur - t_next).item()
            t_batch = torch.full((1,), t_cur, device=device, dtype=torch.bfloat16)
            out     = bridge(z_t[:, :inf_len, :], t_batch, z_acc_crop)
            z_hat_crop = recover_x0(z_t[:, :inf_len, :], out, t_cur, SIGMA_MAX, PARAMETERIZATION) \
                         if PARAMETERIZATION != 'cfm' else z_t[:, :inf_len, :] - out * dt
            z_nat_hat = z_t.clone()
            z_nat_hat[:, :inf_len, :] = z_hat_crop
            if i == N_STEPS - 1:
                break
            if PARAMETERIZATION == 'cfm':
                z_t[:, :inf_len, :] = z_hat_crop
            else:
                z_t = (1 - t_next / t_cur) * z_nat_hat + (t_next / t_cur) * z_t
    return z_nat_hat.squeeze(0).float().cpu()

def speech_cos_local(a, b, T):
    n = min(T, a.shape[0], b.shape[0])
    return F.cosine_similarity(a[:n], b[:n], dim=-1).mean().item()

T_cmp = min(T_l2, T_nat_gt)

z_hat_normal    = run_inference_with_z_t_init(z_acc_bf16)
z_hat_random    = run_inference_with_z_t_init(torch.randn_like(z_acc_bf16))
z_hat_wrong_utt = run_inference_with_z_t_init(z_acc_other_bf16)

print()
print('=== z_t sensitivity test ===')
print('If model uses z_t:    random/wrong outputs should DIVERGE from normal')
print('If model ignores z_t: all three outputs should be ~identical\n')
print(f'{"run":<22}  {"vs normal":>10}  {"vs z_nat":>10}')
print('-' * 48)
r_rn = speech_cos_local(z_hat_random,    z_hat_normal, T_cmp)
r_wn = speech_cos_local(z_hat_wrong_utt, z_hat_normal, T_cmp)
print(f'{"normal":<22}  {1.0:>10.4f}  {speech_cos_local(z_hat_normal, z_nat, T_cmp):>10.4f}')
print(f'{"random z_t":<22}  {r_rn:>10.4f}  {speech_cos_local(z_hat_random,    z_nat, T_cmp):>10.4f}')
print(f'{"wrong utterance":<22}  {r_wn:>10.4f}  {speech_cos_local(z_hat_wrong_utt, z_nat, T_cmp):>10.4f}')
print()
print('Verdict:')
print(f'  random z_t vs normal    = {r_rn:.4f}  {"-> IGNORES z_t" if r_rn > 0.98 else "-> uses z_t"}')
print(f'  wrong_utt z_t vs normal = {r_wn:.4f}  {"-> IGNORES z_t" if r_wn > 0.98 else "-> uses z_t"}')

Other utterance: arctic_a0031  T_l2=123
This utterance:  arctic_a0313  T_l2=193

=== z_t sensitivity test ===
If model uses z_t:    random/wrong outputs should DIVERGE from normal
If model ignores z_t: all three outputs should be ~identical

run                      vs normal    vs z_nat
------------------------------------------------
normal                      1.0000      0.5218
random z_t                  0.0426      0.0796
wrong utterance             0.3945      0.4509

Verdict:
  random z_t vs normal    = 0.0426  -> uses z_t
  wrong_utt z_t vs normal = 0.3945  -> uses z_t
